# Keyword Co-occurrence Networks (KCNs) in Scientific Publications

This notebook constructs and analyses keyword co-occurrence networks (KCNs) derived from a representative sample of academic publication metadata. By linking keywords that appear together within the same paper, KCNs reveal thematic clusters, bridging concepts, and the evolving structure of a research field.

## 1. Setup and Imports

In [1]:
import itertools
import collections

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')          # non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Reproducibility
np.random.seed(42)
print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Dataset

We use a curated sample of 30 fictional but representative paper records spanning four research themes: **Machine Learning**, **Natural Language Processing**, **Network Science**, and **Bioinformatics**. Each record contains a title and a list of author-supplied keywords.

In [2]:
papers = [
    # Machine Learning cluster
    {'id': 1,  'title': 'Deep learning for image recognition',          'keywords': ['deep learning', 'neural networks', 'image recognition', 'convolutional networks']},
    {'id': 2,  'title': 'Transfer learning in NLP',                     'keywords': ['transfer learning', 'deep learning', 'NLP', 'BERT']},
    {'id': 3,  'title': 'Reinforcement learning for robotics',           'keywords': ['reinforcement learning', 'robotics', 'neural networks', 'deep learning']},
    {'id': 4,  'title': 'Generative adversarial networks survey',        'keywords': ['deep learning', 'generative models', 'neural networks', 'GANs']},
    {'id': 5,  'title': 'Federated learning privacy',                    'keywords': ['federated learning', 'deep learning', 'privacy', 'distributed learning']},
    {'id': 6,  'title': 'Graph neural networks for classification',       'keywords': ['graph neural networks', 'deep learning', 'node classification', 'GNNs']},
    {'id': 7,  'title': 'Attention mechanisms in transformers',           'keywords': ['attention mechanisms', 'deep learning', 'transformers', 'BERT']},
    # NLP cluster
    {'id': 8,  'title': 'Sentiment analysis with transformers',          'keywords': ['sentiment analysis', 'NLP', 'transformers', 'BERT']},
    {'id': 9,  'title': 'Named entity recognition survey',               'keywords': ['named entity recognition', 'NLP', 'information extraction', 'sequence labeling']},
    {'id': 10, 'title': 'Machine translation using attention',           'keywords': ['machine translation', 'NLP', 'attention mechanisms', 'sequence-to-sequence']},
    {'id': 11, 'title': 'Text summarization with BERT',                  'keywords': ['text summarization', 'NLP', 'BERT', 'abstractive summarization']},
    {'id': 12, 'title': 'Question answering with large language models', 'keywords': ['question answering', 'NLP', 'large language models', 'BERT']},
    {'id': 13, 'title': 'Coreference resolution in discourse',           'keywords': ['coreference resolution', 'NLP', 'discourse analysis', 'information extraction']},
    # Network Science cluster
    {'id': 14, 'title': 'Community detection algorithms',                'keywords': ['community detection', 'graph theory', 'social networks', 'clustering algorithms']},
    {'id': 15, 'title': 'Link prediction in knowledge graphs',           'keywords': ['link prediction', 'knowledge graphs', 'graph theory', 'graph neural networks']},
    {'id': 16, 'title': 'Network centrality measures survey',            'keywords': ['centrality measures', 'graph theory', 'social networks', 'network analysis']},
    {'id': 17, 'title': 'Scale-free networks and power-law distributions','keywords': ['scale-free networks', 'graph theory', 'power-law', 'network analysis']},
    {'id': 18, 'title': 'Epidemic spreading on complex networks',        'keywords': ['epidemic models', 'complex networks', 'graph theory', 'social networks']},
    {'id': 19, 'title': 'Temporal networks analysis',                    'keywords': ['temporal networks', 'network analysis', 'graph theory', 'dynamic graphs']},
    {'id': 20, 'title': 'Multilayer networks review',                    'keywords': ['multilayer networks', 'complex networks', 'network analysis', 'graph theory']},
    # Bioinformatics cluster
    {'id': 21, 'title': 'Protein structure prediction with deep learning','keywords': ['protein structure', 'deep learning', 'bioinformatics', 'AlphaFold']},
    {'id': 22, 'title': 'Gene expression analysis via ML',               'keywords': ['gene expression', 'machine learning', 'bioinformatics', 'RNA-seq']},
    {'id': 23, 'title': 'Drug-target interaction prediction',            'keywords': ['drug-target interaction', 'graph neural networks', 'bioinformatics', 'deep learning']},
    {'id': 24, 'title': 'Single-cell RNA sequencing methods',            'keywords': ['single-cell analysis', 'RNA-seq', 'bioinformatics', 'gene expression']},
    {'id': 25, 'title': 'CRISPR guide design with ML',                   'keywords': ['CRISPR', 'machine learning', 'bioinformatics', 'deep learning']},
    # Cross-cutting papers
    {'id': 26, 'title': 'NLP for biomedical text mining',                'keywords': ['NLP', 'bioinformatics', 'text mining', 'named entity recognition']},
    {'id': 27, 'title': 'Knowledge graph embeddings for biology',        'keywords': ['knowledge graphs', 'bioinformatics', 'graph neural networks', 'embeddings']},
    {'id': 28, 'title': 'Social network analysis of citation networks',  'keywords': ['social networks', 'network analysis', 'citation networks', 'bibliometrics']},
    {'id': 29, 'title': 'Explainability in deep learning',               'keywords': ['explainability', 'deep learning', 'neural networks', 'fairness']},
    {'id': 30, 'title': 'Automated machine learning (AutoML)',           'keywords': ['AutoML', 'machine learning', 'deep learning', 'hyperparameter optimization']},
]

df = pd.DataFrame(papers)
print(f'Dataset: {len(df)} papers')
print(f'Unique keywords: {len(set(k for kws in df.keywords for k in kws))}')
df.head()

Dataset: 30 papers
Unique keywords: 63


,id,title,keywords
0,1,Deep learning for image recognition,"[deep learning, neural networks, image recogni..."
1,2,Transfer learning in NLP,"[transfer learning, deep learning, NLP, BERT]"
2,3,Reinforcement learning for robotics,"[reinforcement learning, robotics, neural netw..."
3,4,Generative adversarial networks survey,"[deep learning, generative models, neural netw..."
4,5,Federated learning privacy,"[federated learning, deep learning, privacy, d..."


## 3. Building the Keyword Co-occurrence Network

For every paper we generate all pairwise keyword combinations. Each pair becomes an edge; repeated pairs increment the edge **weight**. The result is an undirected weighted graph $G=(V,E)$ where:
- **Nodes** $V$ represent unique keywords.
- **Edges** $E$ represent co-occurrence; $w(u,v)$ is the number of papers containing both $u$ and $v$.

In [3]:
G = nx.Graph()

for _, row in df.iterrows():
    keywords = row['keywords']
    for kw in keywords:
        if not G.has_node(kw):
            G.add_node(kw, count=0)
        G.nodes[kw]['count'] += 1
    for kw1, kw2 in itertools.combinations(keywords, 2):
        if G.has_edge(kw1, kw2):
            G[kw1][kw2]['weight'] += 1
        else:
            G.add_edge(kw1, kw2, weight=1)

print(f'Nodes (unique keywords): {G.number_of_nodes()}')
print(f'Edges (co-occurrence pairs): {G.number_of_edges()}')
print(f'Network density: {nx.density(G):.4f}')

Nodes (unique keywords): 63
Edges (co-occurrence pairs): 153
Network density: 0.0783


## 4. Descriptive Network Statistics

We compute basic structural metrics to characterise the network before visualisation.

In [4]:
degrees = dict(G.degree())
weighted_degrees = dict(G.degree(weight='weight'))
degree_values = list(degrees.values())

print('=== Basic Statistics ===')
print(f'  Nodes                  : {G.number_of_nodes()}')
print(f'  Edges                  : {G.number_of_edges()}')
print(f'  Density                : {nx.density(G):.4f}')
print(f'  Average degree         : {np.mean(degree_values):.2f}')
print(f'  Max degree             : {max(degree_values)}')
print(f'  Min degree             : {min(degree_values)}')
print(f'  Is connected           : {nx.is_connected(G)}')

if nx.is_connected(G):
    print(f'  Avg clustering coeff   : {nx.average_clustering(G, weight="weight"):.4f}')
    print(f'  Avg shortest path len  : {nx.average_shortest_path_length(G):.4f}')
else:
    largest_cc = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    print(f'  Connected components   : {nx.number_connected_components(G)}')
    print(f'  Largest CC size        : {largest_cc.number_of_nodes()} nodes')
    print(f'  Avg clustering coeff   : {nx.average_clustering(G, weight="weight"):.4f}')
    print(f'  Avg shortest path (LCC): {nx.average_shortest_path_length(largest_cc):.4f}')

=== Basic Statistics ===
  Nodes                  : 63
  Edges                  : 153
  Density                : 0.0783
  Average degree         : 4.86
  Max degree             : 28
  Min degree             : 3
  Is connected           : True
  Avg clustering coeff   : 0.2523
  Avg shortest path len  : 2.9186


## 5. Centrality Analysis

Three complementary centrality measures identify the most influential keywords:

| Measure | What it captures |
|---------|------------------|
| **Degree centrality** | How many distinct keywords this keyword co-occurs with |
| **Betweenness centrality** | How often a keyword lies on shortest paths (bridges) |
| **PageRank** | Recursive importance (influential neighbours raise score) |

In [5]:
degree_centrality     = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G, weight='weight', normalized=True)
pagerank              = nx.pagerank(G, weight='weight')

centrality_df = pd.DataFrame({
    'keyword'      : list(G.nodes()),
    'degree'       : [degrees[n] for n in G.nodes()],
    'degree_cent'  : [degree_centrality[n] for n in G.nodes()],
    'betweenness'  : [betweenness_centrality[n] for n in G.nodes()],
    'pagerank'     : [pagerank[n] for n in G.nodes()],
    'paper_count'  : [G.nodes[n]['count'] for n in G.nodes()],
}).sort_values('degree_cent', ascending=False).reset_index(drop=True)

print('Top 10 keywords by degree centrality:')
centrality_df.head(10)

Top 10 keywords by degree centrality:


,keyword,degree,degree_cent,betweenness,pagerank,paper_count
0,deep learning,28,0.451613,0.474740,0.091231,12
1,NLP,19,0.306452,0.311258,0.060879,8
2,graph theory,15,0.241935,0.369708,0.052905,7
3,bioinformatics,15,0.241935,0.300964,0.050690,7
4,network analysis,11,0.177419,0.044433,0.038750,5
5,BERT,10,0.161290,0.004848,0.037722,5
6,neural networks,9,0.145161,0.006346,0.032663,4
7,social networks,9,0.145161,0.028085,0.031278,4
8,graph neural networks,9,0.145161,0.189612,0.030205,4
9,machine learning,7,0.112903,0.024412,0.022986,3


## 6. Community Detection

We apply the **Louvain** community-detection algorithm (greedy modularity maximisation) to uncover thematic clusters within the network. Each community corresponds to a coherent research sub-topic.

In [6]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G, weight='weight'))
communities.sort(key=len, reverse=True)

print(f'Number of communities detected: {len(communities)}')
for i, comm in enumerate(communities):
    print(f'\nCommunity {i+1} ({len(comm)} keywords):')
    print('  ' + ', '.join(sorted(comm)))

# Assign community labels to nodes
community_map = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_map[node] = i
nx.set_node_attributes(G, community_map, 'community')

Number of communities detected: 5

Community 1 (18 keywords):
  BERT, NLP, abstractive summarization, attention mechanisms, coreference resolution, discourse analysis, information extraction, large language models, machine translation, named entity recognition, question answering, sentiment analysis, sequence labeling, sequence-to-sequence, text mining, text summarization, transfer learning, transformers

Community 2 (15 keywords):
  bibliometrics, centrality measures, citation networks, clustering algorithms, community detection, complex networks, dynamic graphs, epidemic models, graph theory, multilayer networks, network analysis, power-law, scale-free networks, social networks, temporal networks

Community 3 (13 keywords):
  GANs, convolutional networks, deep learning, distributed learning, explainability, fairness, federated learning, generative models, image recognition, neural networks, privacy, reinforcement learning, robotics

Community 4 (10 keywords):
  AlphaFold, AutoML, CRI

## 7. Network Visualisation

We produce two complementary plots:
1. **Full KCN** – all nodes coloured by community, sized by degree centrality.
2. **Backbone KCN** – only edges with weight ≥ 2 to highlight strong co-occurrences.

In [7]:
PALETTE = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3',
           '#ff7f00', '#a65628', '#f781bf', '#999999']

def draw_kcn(graph, title, ax, edge_thresh=1):
    filtered = graph.edge_subgraph(
        [(u, v) for u, v, d in graph.edges(data=True) if d['weight'] >= edge_thresh]
    ).copy()
    if filtered.number_of_nodes() == 0:
        ax.set_title(title)
        ax.axis('off')
        return

    pos = nx.spring_layout(filtered, seed=42, k=0.6)

    node_colors = [PALETTE[filtered.nodes[n].get('community', 0) % len(PALETTE)]
                   for n in filtered.nodes()]
    dc = nx.degree_centrality(filtered)
    node_sizes  = [600 + 3000 * dc[n] for n in filtered.nodes()]
    edge_widths = [1 + d['weight'] for u, v, d in filtered.edges(data=True)]

    nx.draw_networkx_edges(filtered, pos, ax=ax,
                           width=edge_widths, alpha=0.35, edge_color='#555555')
    nx.draw_networkx_nodes(filtered, pos, ax=ax,
                           node_color=node_colors, node_size=node_sizes, alpha=0.9)
    nx.draw_networkx_labels(filtered, pos, ax=ax, font_size=7, font_weight='bold')

    n_comm = max(filtered.nodes[n].get('community', 0) for n in filtered.nodes()) + 1
    patches = [mpatches.Patch(color=PALETTE[i % len(PALETTE)], label=f'Community {i+1}')
               for i in range(n_comm)]
    ax.legend(handles=patches, loc='upper left', fontsize=7, framealpha=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')


fig, axes = plt.subplots(1, 2, figsize=(18, 9))
draw_kcn(G, 'Full Keyword Co-occurrence Network', axes[0], edge_thresh=1)
draw_kcn(G, 'Backbone KCN (edge weight ≥ 2)',     axes[1], edge_thresh=2)
plt.tight_layout()
plt.savefig('kcn_visualisation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved as kcn_visualisation.png')

Figure saved as kcn_visualisation.png


## 8. Top Keywords by Centrality

The bar charts below rank the 15 most central keywords under each metric, providing a quick at-a-glance summary of network hubs.

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = [
    ('degree_cent', 'Degree Centrality',     '#377eb8'),
    ('betweenness', 'Betweenness Centrality', '#e41a1c'),
    ('pagerank',    'PageRank',               '#4daf4a'),
]

for ax, (col, label, color) in zip(axes, metrics):
    top = centrality_df.nlargest(15, col)
    ax.barh(top['keyword'][::-1], top[col][::-1], color=color, alpha=0.85)
    ax.set_xlabel(label, fontsize=11)
    ax.set_title(f'Top 15 — {label}', fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('kcn_centrality.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved as kcn_centrality.png')

Figure saved as kcn_centrality.png


## 9. Edge-Weight Distribution

The distribution of co-occurrence weights shows whether the network is dominated by weak ties (weight = 1) or contains a meaningful core of strongly co-occurring keyword pairs.

In [9]:
weights = [d['weight'] for _, _, d in G.edges(data=True)]
weight_counts = collections.Counter(weights)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(weight_counts.keys(), weight_counts.values(), color='#984ea3', alpha=0.85, edgecolor='white')
ax.set_xlabel('Co-occurrence Weight', fontsize=12)
ax.set_ylabel('Number of Edges', fontsize=12)
ax.set_title('Distribution of Edge Weights in the KCN', fontsize=13, fontweight='bold')
ax.set_xticks(sorted(weight_counts.keys()))
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('kcn_edge_weights.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Weight distribution: {dict(sorted(weight_counts.items()))}')

Weight distribution: {1: 134, 2: 14, 3: 2, 4: 3}


## 10. Summary Statistics Table

A consolidated view of the top-ranked keywords across all three centrality measures.

In [10]:
summary = centrality_df[['keyword', 'degree', 'degree_cent', 'betweenness', 'pagerank', 'paper_count']].head(15)
summary.columns = ['Keyword', 'Degree', 'Degree Cent.', 'Betweenness', 'PageRank', 'Paper Count']
summary = summary.round({'Degree Cent.': 4, 'Betweenness': 4, 'PageRank': 4})
summary

,Keyword,Degree,Degree Cent.,Betweenness,PageRank,Paper Count
0,deep learning,28,0.4516,0.4747,0.0912,12
1,NLP,19,0.3065,0.3113,0.0609,8
2,graph theory,15,0.2419,0.3697,0.0529,7
3,bioinformatics,15,0.2419,0.3010,0.0507,7
4,network analysis,11,0.1774,0.0444,0.0387,5
5,BERT,10,0.1613,0.0048,0.0377,5
6,neural networks,9,0.1452,0.0063,0.0327,4
7,social networks,9,0.1452,0.0281,0.0313,4
8,graph neural networks,9,0.1452,0.1896,0.0302,4
9,machine learning,7,0.1129,0.0244,0.0230,3


## 11. Conclusions

This notebook demonstrated the full KCN pipeline:

1. **Data preparation** – converting keyword lists from paper records into a structured dataset.
2. **Graph construction** – building a weighted undirected co-occurrence graph.
3. **Structural analysis** – computing density, clustering, and path-length metrics.
4. **Centrality ranking** – identifying hub keywords via degree, betweenness, and PageRank.
5. **Community detection** – discovering thematic clusters using greedy modularity maximisation.
6. **Visualisation** – producing full and backbone network plots and centrality bar charts.

Key findings are interpreted in detail in `report.md`.